# 🛰️ Starlink Orbit Astrodynamics & SGP4 Propagation
## Interactive Engineering Notebook: J2 Nodal Precession & Walker Delta Simulation

> **Lead Disciplines:** `spacecraft-orbital-mechanics-engineer` & `aerospace-flight-mathematics-expert`  
> **Standards Reference:** StarSec CSF v1.0, Technical Monograph Chapter 04 & R&D Whitepaper 01

This notebook provides interactive astrodynamics tools to model Low Earth Orbit (LEO) mega-constellation orbital geometry, secular $J_2$ Earth oblateness perturbations, and orbital ground tracks.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Ensure local simulations module is importable
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root))

from simulations.orbit_sgp4_propagator import (
    OrbitalElements,
    ConjunctionGeometry,
    calculate_j2_secular_precession,
    keplerian_to_cartesian_state,
    generate_walker_delta_constellation,
    foster_collision_probability,
    MU_EARTH,
    RADIUS_EARTH_KM,
    J2_COEFFICIENT
)

print("✅ Successfully imported Starlink Astrodynamics Simulation Suite")

### 1. J2 Secular Nodal Regression Rate ($\\dot{\\Omega}$)

Due to Earth's equatorial bulge (governed by second zonal gravitational harmonic $J_2 = 1.08263 \times 10^{-3}$), the orbital plane precesses according to:

$$\dot{\Omega} = -\frac{3}{2} J_2 \left(\frac{R_E}{p}\right)^2 n \cos(i)$$

For Starlink Shell 1 ($h = 550\text{ km}, i = 53.0^\circ$), the nodal precession rate is $\approx -4.49^\circ/\text{day}$.

In [ ]:
elements_shell1 = OrbitalElements(
    semi_major_axis_km=RADIUS_EARTH_KM + 550.0,
    eccentricity=0.0001,
    inclination_deg=53.0,
    raan_deg=0.0,
    arg_perigee_deg=0.0,
    true_anomaly_deg=0.0
)

raan_rate_deg_day, perigee_rate_deg_day = calculate_j2_secular_precession(elements_shell1)
print(f"Starlink Shell 1 (550 km, 53.0°):")
print(f"  • RAAN Precession Rate (dΩ/dt)     : {raan_rate_deg_day:.3f} deg/day")
print(f"  • Perigee Precession Rate (dω/dt) : {perigee_rate_deg_day:.3f} deg/day")

# Sweep inclination across orbits
inclinations = np.linspace(0, 180, 100)
raan_rates = [
    calculate_j2_secular_precession(
        OrbitalElements(RADIUS_EARTH_KM + 550.0, 0.0001, inc, 0.0, 0.0, 0.0)
    )[0] for inc in inclinations
]

plt.figure(figsize=(9, 4.5))
plt.plot(inclinations, raan_rates, 'b-', lw=2, label="550 km LEO Shell")
plt.axvline(53.0, color='r', linestyle='--', label="Starlink Shell 1 (53.0°)")
plt.axvline(97.6, color='g', linestyle='--', label="Sun-Synchronous (97.6°)")
plt.axhline(0, color='black', lw=0.8, linestyle=':')
plt.title("J2 Nodal Precession Rate vs Orbital Inclination")
plt.xlabel("Inclination (degrees)")
plt.ylabel("RAAN Drift Rate (deg/day)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

### 2. Walker Delta Constellation Synthesis ($53^\circ: 72/22/1$)

Starlink Shell 1 deploys 72 orbital planes with 22 satellites per plane (total 1,584 spacecraft) phased symmetrically to provide continuous global coverage.

In [ ]:
constellation = generate_walker_delta_constellation(
    total_satellites=72 * 22,
    num_planes=72,
    phasing_parameter=1,
    altitude_km=550.0,
    inclination_deg=53.0
)

print(f"Synthesized Walker Delta Shell: {len(constellation)} spacecraft across 72 orbital planes.")

# Sample 3D orbital positions for first 8 planes
fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(111, projection='3d')

for sat in constellation[:72 * 4]:
    pos, _ = keplerian_to_cartesian_state(sat)
    ax.scatter(pos[0], pos[1], pos[2], color='navy', s=4, alpha=0.6)

ax.set_title("Starlink Shell 1 Walker Delta 3D Spatial Mesh")
ax.set_xlabel("X (km)")
ax.set_ylabel("Y (km)")
ax.set_zlabel("Z (km)")
plt.tight_layout()
plt.show()

### 3. Autonomous Collision Probability Evaluation (Foster Integral)

Autonomous conjunction avoidance integrates positional covariance ellipses in the B-plane to determine collision risk against orbital debris.

In [ ]:
conjunction = ConjunctionGeometry(
    miss_distance_km=0.150,      # 150 meters miss distance
    sigma_x_km=0.080,            # 80 meters along-track error
    sigma_y_km=0.040,            # 40 meters cross-track error
    correlation_rho=0.15,
    combined_radius_m=10.0       # 10m hard-body sphere
)

pc = foster_collision_probability(conjunction)
print(f"Conjunction Encounter Analysis:")
print(f"  • Miss Distance                : {conjunction.miss_distance_km*1000:.1f} m")
print(f"  • Foster Collision Probability : {pc:.2e}")
print(f"  • Avoidance Threshold (1.0e-4) : {'MANEUVER TRIGGERED' if pc > 1e-4 else 'MONITOR ONLY'}")